# Vectorización de Texto: Métodos Frecuentistas
## UNR · TUIA · Procesamiento de Lenguaje Natural — Unidad 2

Para que un modelo pueda trabajar con texto, primero hay que convertirlo en números.
En esta práctica exploramos los métodos clásicos con un dataset real de libros.

| Parte | Tema | Ejercicios |
|-------|------|-----------|
| 1 | Preparación del dataset | Ej 1, 2 |
| 2 | One-Hot Encoding | Ej 3 |
| 3 | Count Vectorizer | Ej 4 |
| 4 | N-grams | — (solo lectura) |
| 5 | TF-IDF | Ej 5 |
| 6 | Hash Vectorizer | — (solo lectura) |
| 7 | Comparación de métodos | Ej 6, 7 |
| 8 | Visualización con PCA | Ej 8 |

> **Texto**: sinopsis (`sinapsis`) · **Etiqueta**: subgénero · **Dataset**: Lectulandia


In [1]:
# ── Dependencias ─────────────────────────────────────────────────────────────
!pip install scikit-learn numpy pandas matplotlib seaborn

import re, warnings
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.sparse import issparse

from sklearn.feature_extraction.text import (
    CountVectorizer, TfidfVectorizer, HashingVectorizer
)
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings("ignore")
plt.rcParams["figure.dpi"] = 110
np.random.seed(42)

print("Imports OK")


Imports OK


In [2]:
# ── Descarga del dataset ──────────────────────────────────────────────────────
import os, urllib.request

DATA_PATH = "data/lectulandia_books.csv"
GDRIVE_ID = "147g4SlXZtguJ7LZ1-9zxaHiYXlTApqru"

os.makedirs(os.path.dirname(DATA_PATH), exist_ok=True)

if not os.path.exists(DATA_PATH):
    print("Descargando lectulandia_books.csv desde Google Drive...")
    try:
        import gdown
        gdown.download(id=GDRIVE_ID, output=DATA_PATH, quiet=False)
    except ImportError:
        # Fallback sin gdown (funciona para archivos públicos sin confirmación)
        url = f"https://drive.google.com/uc?export=download&id={GDRIVE_ID}"
        urllib.request.urlretrieve(url, DATA_PATH)
    print(f"Dataset guardado en: {DATA_PATH}")
else:
    print(f"Dataset ya existe en: {DATA_PATH}")


Descargando lectulandia_books.csv desde Google Drive...
Dataset guardado en: data/lectulandia_books.csv


---
# Parte 1 – Preparación del Dataset

## El dataset: Lectulandia

Tenemos un CSV con ~476 libros. Cada fila tiene:

| Columna | Ejemplo |
|---------|---------|
| `titulo` | "El año del pensamiento mágico" |
| `autor` | "Joan Didion" |
| `sinapsis` | Texto libre de ~200 palabras |
| `generos` | "Biografía - Crónica - Literatura" |

El campo `generos` puede tener varios valores separados por ` - `. Vamos a usar el
**segundo género** como etiqueta de clase (es el más discriminativo).

## ¿Por qué balancear?

Los tres subgéneros más frecuentes tienen muy distinta cantidad de libros:
Crónica tiene ~169, Ensayo ~78 y Divulgación ~59.

Si dejamos ese desbalance, el modelo aprende a predecir siempre la clase mayoritaria
y parece bueno sin aprender nada útil. Por eso tomamos la misma cantidad de libros
de cada clase.

## Limpieza del texto

Antes de vectorizar, limpiamos cada sinopsis:

```python
import re

def limpiar_texto(texto):
    texto = texto.lower()                       # minúsculas
    texto = re.sub(r"[^\w\s]", " ", texto)    # sin puntuación
    texto = re.sub(r"\d+", " ", texto)         # sin números
    texto = re.sub(r"\s+", " ", texto).strip() # sin espacios dobles
    return texto
```


### Ejercicio 1 – Preparar el Dataset

In [10]:
import re

def limpiar_texto(texto):
    texto = texto.lower()                       # minúsculas
    texto = re.sub(r"[^\w\s]", " ", texto)    # sin puntuación
    texto = re.sub(r"\d+", " ", texto)         # sin números
    texto = re.sub(r"\s+", " ", texto).strip() # sin espacios dobles
    return texto

In [ ]:
def preparar_dataset(path):
    """
    TODO:
    1. Cargar el CSV y eliminar filas con sinapsis o géneros vacíos
    2. Extraer el 2° género como etiqueta ('label')
    3. Quedarse con los top-3 géneros y balancear (random_state=42)
    4. Limpiar la sinopsis → columna 'sinapsis_clean'
       (minúsculas, sin puntuación, sin números)
    """
    pass

# df = preparar_dataset(DATA_PATH)

df = pd.read_csv(DATA_PATH)
# me quedo con las columnas titulo, autor, sinapsis y generos
df = df[['titulo', 'autor', 'sinapsis', 'generos']]
df.head()

,titulo,autor,sinapsis,generos
0,Tierra feroz,Jota Quijorna,La oscuridad de un alma herida es el hogar de ...,Intriga - Novela
1,Tehanu,Ursula K. Le Guin,"El mal medra, y la magia se ha pervertido. En ...",Fantástico - Novela
2,El niño del taxi,Sylvain Prudhomme,"Durante el funeral de su abuelo, Simon descubr...",Histórico - Novela
3,James Bond 007: El juego de rol,Gerard Christopher Klug,ATRÉVETE A VIVIR EN EL FASCINANTE MUNDO DE JAM...,Juegos - Manuales y cursos - Referencia
4,El retorno del cuervo,Alissa Brontë,"Tras varios años alejado del que fue su hogar,...",Histórico - Novela - Romántico


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 62279 entries, 0 to 62278
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   titulo    62279 non-null  object
 1   autor     62278 non-null  object
 2   sinapsis  47830 non-null  object
 3   generos   62278 non-null  object
dtypes: object(4)
memory usage: 1.9+ MB


In [9]:
# eliminar filas con sinapsis o géneros vacíos
df = df.dropna(subset=['sinapsis', 'generos'])
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 47830 entries, 0 to 61942
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   titulo    47830 non-null  object
 1   autor     47829 non-null  object
 2   sinapsis  47830 non-null  object
 3   generos   47830 non-null  object
dtypes: object(4)
memory usage: 1.8+ MB


In [ ]:
# 2. Extraer el 2° género como etiqueta ('label')
# Crónica - Memorias
# Arqueología - Ciencias sociales - Divulgación - Historia
df['label'] = df['generos'].apply(lambda x: x.split('-')[1] if len(x.split('-')) > 1 else None)
df.head()

,titulo,autor,sinapsis,generos,label
0,Tierra feroz,Jota Quijorna,La oscuridad de un alma herida es el hogar de ...,Intriga - Novela,Novela
1,Tehanu,Ursula K. Le Guin,"El mal medra, y la magia se ha pervertido. En ...",Fantástico - Novela,Novela
2,El niño del taxi,Sylvain Prudhomme,"Durante el funeral de su abuelo, Simon descubr...",Histórico - Novela,Novela
3,James Bond 007: El juego de rol,Gerard Christopher Klug,ATRÉVETE A VIVIR EN EL FASCINANTE MUNDO DE JAM...,Juegos - Manuales y cursos - Referencia,Manuales y cursos
4,El retorno del cuervo,Alissa Brontë,"Tras varios años alejado del que fue su hogar,...",Histórico - Novela - Romántico,Novela


In [13]:
# Chequear si hay label vacías
print(df['label'].isna().sum())

53


In [14]:
# 3. Quedarse con los top-3 géneros y balancear (random_state=42)
df['label'].value_counts()

label
Novela           10831
Novela            5803
Relato            2704
Policíaco         2344
Otros             2265
                 ...  
Tecnolog√≠a          1
Religión             1
Recopilación         1
Cuentos              1
Esoterismo           1
Name: count, Length: 162, dtype: int64

In [15]:
# normalizar labels
df['label'] = df['label'].str.strip().str.lower()
df['label'].value_counts()

label
novela          16634
relato           2892
otros            2704
ensayo           2623
historia         2569
                ...  
magia               1
religión            1
recopilación        1
cuentos             1
esoterismo          1
Name: count, Length: 111, dtype: int64

In [16]:
# 4. Limpiar la sinopsis → columna 'sinapsis_clean' (minúsculas, sin puntuación, sin números)
df['sinapsis_clean'] = df['sinapsis'].apply(limpiar_texto)
df.head()

,titulo,autor,sinapsis,generos,label,sinapsis_clean
0,Tierra feroz,Jota Quijorna,La oscuridad de un alma herida es el hogar de ...,Intriga - Novela,novela,la oscuridad de un alma herida es el hogar de ...
1,Tehanu,Ursula K. Le Guin,"El mal medra, y la magia se ha pervertido. En ...",Fantástico - Novela,novela,el mal medra y la magia se ha pervertido en to...
2,El niño del taxi,Sylvain Prudhomme,"Durante el funeral de su abuelo, Simon descubr...",Histórico - Novela,novela,durante el funeral de su abuelo simon descubre...
3,James Bond 007: El juego de rol,Gerard Christopher Klug,ATRÉVETE A VIVIR EN EL FASCINANTE MUNDO DE JAM...,Juegos - Manuales y cursos - Referencia,manuales y cursos,atrévete a vivir en el fascinante mundo de jam...
4,El retorno del cuervo,Alissa Brontë,"Tras varios años alejado del que fue su hogar,...",Histórico - Novela - Romántico,novela,tras varios años alejado del que fue su hogar ...


### Ejercicio 2 – Explorar el Corpus

In [ ]:
def explorar_corpus(df):
    """
    TODO:
    1. Mostrar cuántos libros hay por género
    2. Graficar la longitud de sinopsis por género (boxplot)
    3. Mostrar las 10 palabras más frecuentes del corpus
       (excluir stopwords básicas como 'el', 'de', 'la', etc.)
    """
    pass

explorar_corpus(df)


---
# Parte 2 – One-Hot Encoding

## ¿Qué hace?

Cada documento se convierte en un vector de 0s y 1s. El vector tiene una posición
por cada palabra del vocabulario: `1` si la palabra aparece, `0` si no.

**Ejemplo** con tres documentos:

```
d1: "el gato come pescado"
d2: "el perro come carne"
d3: "el gato y el perro juegan"
```

| doc | carne | come | el | gato | juegan | perro | pescado | y |
|-----|:-----:|:----:|:--:|:----:|:------:|:-----:|:-------:|:-:|
| d1  |   0   |   1  |  1 |   1  |   0    |   0   |    1    | 0 |
| d2  |   1   |   1  |  1 |   0  |   0    |   1   |    0    | 0 |
| d3  |   0   |   0  |  1 |   1  |   1    |   1   |    0    | 1 |

En sklearn se obtiene con `CountVectorizer(binary=True)`.

## El problema: demasiados ceros

Si el vocabulario tiene 10 000 palabras y una sinopsis usa ~200, el vector tiene
9 800 ceros → **98% de ceros**. A esto lo llamamos **sparsidad**.

Esto es normal y manejable (sklearn usa formatos especiales que no guardan los ceros),
pero vale la pena entender cómo crece con el tamaño del vocabulario.


In [ ]:
# ── Ejemplo interactivo: One-Hot Encoding ────────────────────────────────────
from sklearn.feature_extraction.text import CountVectorizer
import pandas as pd

corpus_demo = [
    "el gato come pescado",
    "el perro come carne",
    "el gato y el perro juegan",
]

cv_ohe = CountVectorizer(binary=True)
X_ohe  = cv_ohe.fit_transform(corpus_demo)

print("=== One-Hot Encoding ===")
print(f"Shape: {X_ohe.shape}  (docs × términos)")
print(f"Sparsidad: {1 - X_ohe.nnz / (X_ohe.shape[0] * X_ohe.shape[1]):.1%}\n")

df_ohe = pd.DataFrame(
    X_ohe.toarray(),
    columns=cv_ohe.get_feature_names_out(),
    index=[f"d{i+1}" for i in range(len(corpus_demo))]
)
print(df_ohe.to_string())
print("\n→ Notar que 'el' en d3 aparece 2 veces, pero el valor es 1 (solo presencia)")


=== One-Hot Encoding ===
Shape: (3, 7)  (docs × términos)
Sparsidad: 42.9%

    carne  come  el  gato  juegan  perro  pescado
d1      0     1   1     1       0      0        1
d2      1     1   1     0       0      1        0
d3      0     0   1     1       1      1        0

→ Notar que 'el' en d3 aparece 2 veces, pero el valor es 1 (solo presencia)


### Ejercicio 3 – One-Hot Encoding

In [ ]:
def analizar_one_hot(df):
    """
    TODO:
    1. Aplicar CountVectorizer(binary=True) sobre 'sinapsis_clean'
    2. Imprimir dimensiones de la matriz y porcentaje de ceros (sparsidad)
    3. Graficar cómo cambia la sparsidad al variar max_features
       en [500, 1000, 2000, 5000, None]
    """
    pass

analizar_one_hot(df)


---
# Parte 3 – Count Vectorizer

## ¿Qué hace?

A diferencia de One-Hot, Count Vectorizer guarda **cuántas veces** aparece cada
palabra (no solo si aparece o no).

```python
doc = ["el gato come come pescado pescado pescado"]

# One-Hot:  [1, 1, 1, 1]  → "pescado" y "come" valen igual que "el"
# Count:    [1, 2, 1, 3]  → "pescado" ×3 y "come" ×2 se capturan
```

## Parámetros útiles

| Parámetro | Para qué sirve | Ejemplo |
|-----------|---------------|---------|
| `max_features` | Limita el vocabulario a las N palabras más frecuentes | `max_features=5000` |
| `min_df` | Ignora palabras que aparecen en muy pocos documentos | `min_df=2` descarta hapax |
| `max_df` | Ignora palabras que aparecen en casi todos los documentos | `max_df=0.9` descarta stopwords |
| `binary` | Si `True`, equivale a One-Hot | `binary=True` |

**Regla práctica**: casi siempre conviene usar `min_df=2` para descartar palabras
únicas que no ayudan a generalizar.


In [ ]:
# ── Ejemplo interactivo: Count Vectorizer ────────────────────────────────────
from sklearn.feature_extraction.text import CountVectorizer
import pandas as pd

corpus_demo = [
    "el gato come come pescado pescado pescado",
    "el perro come carne",
    "el gato y el perro juegan",
]

# Comparar binary=False vs binary=True
cv_count  = CountVectorizer(binary=False)
cv_binary = CountVectorizer(binary=True)

X_count  = cv_count.fit_transform(corpus_demo)
X_binary = cv_binary.fit_transform(corpus_demo)

idx = [f"d{i+1}" for i in range(len(corpus_demo))]

print("=== CountVectorizer (binary=False) — conteo de frecuencias ===")
df_c = pd.DataFrame(X_count.toarray(),
                    columns=cv_count.get_feature_names_out(), index=idx)
print(df_c.to_string())

print("\n=== CountVectorizer (binary=True) — solo presencia ===")
df_b = pd.DataFrame(X_binary.toarray(),
                    columns=cv_binary.get_feature_names_out(), index=idx)
print(df_b.to_string())

print("\n→ En d1: 'come'×2 y 'pescado'×3 se capturan con binary=False pero no con binary=True")

# Efecto de min_df y max_df
print("\n=== Efecto de min_df=2 (filtra palabras que aparecen en un solo doc) ===")
cv_mindf = CountVectorizer(min_df=2)
X_mindf  = cv_mindf.fit_transform(corpus_demo)
print("Vocabulario:", list(cv_mindf.get_feature_names_out()))


=== CountVectorizer (binary=False) — conteo de frecuencias ===
    carne  come  el  gato  juegan  perro  pescado
d1      0     2   1     1       0      0        3
d2      1     1   1     0       0      1        0
d3      0     0   2     1       1      1        0

=== CountVectorizer (binary=True) — solo presencia ===
    carne  come  el  gato  juegan  perro  pescado
d1      0     1   1     1       0      0        1
d2      1     1   1     0       0      1        0
d3      0     0   1     1       1      1        0

→ En d1: 'come'×2 y 'pescado'×3 se capturan con binary=False pero no con binary=True

=== Efecto de min_df=2 (filtra palabras que aparecen en un solo doc) ===
Vocabulario: ['come', 'el', 'gato', 'perro']


### Ejercicio 4 – Count Vectorizer: `min_df` y `max_df`

In [ ]:
def explorar_count_vectorizer(df):
    """
    TODO:
    1. Probar min_df en [1, 2, 3, 5, 10] — registrar cuántas features quedan
    2. Probar max_df en [1.0, 0.9, 0.8, 0.7, 0.5] — registrar cuántas features quedan
    3. Graficar n_features vs cada parámetro
    """
    pass

explorar_count_vectorizer(df)


---
# Parte 4 – N-grams

> Esta sección es **solo lectura**: entendé el concepto y ejecutá el ejemplo.
> Los n-grams los vas a usar en el Ejercicio 6 (benchmark).

## ¿Qué son?

Un **n-gram** es una secuencia de n palabras consecutivas.

| Tipo | Ejemplo sobre "primera guerra mundial" |
|------|----------------------------------------|
| Unigram (1) | "primera" · "guerra" · "mundial" |
| Bigram (2) | "primera guerra" · "guerra mundial" |
| Trigram (3) | "primera guerra mundial" |

## ¿Para qué sirven?

Los unigramas pierden el contexto. "no me gustó" tratado palabra a palabra puede
confundirse con "me gustó" porque comparten "me" y "gustó".

El bigrama "no me" o "no gustó" captura esa negación.

En sklearn se controla con `ngram_range=(min_n, max_n)`:

```python
# Solo palabras sueltas
CountVectorizer(ngram_range=(1, 1))

# Palabras sueltas + pares de palabras
CountVectorizer(ngram_range=(1, 2))

# Solo pares
CountVectorizer(ngram_range=(2, 2))
```

> Para clasificación de texto, `ngram_range=(1, 2)` suele mejorar los resultados
> respecto a usar solo unigramas, sin agregar demasiada complejidad.


In [ ]:
# ── Ejemplo interactivo: N-grams ─────────────────────────────────────────────
from sklearn.feature_extraction.text import CountVectorizer

corpus_demo = [
    "la primera guerra mundial fue terrible",
    "la segunda guerra mundial también fue terrible",
    "el libro sobre la guerra es excelente",
]

configs = {
    "(1,1) — solo unigrams":   (1, 1),
    "(1,2) — uni + bigrams":   (1, 2),
    "(2,2) — solo bigrams":    (2, 2),
    "(1,3) — uni+bi+trigrams": (1, 3),
}

for nombre, rango in configs.items():
    cv  = CountVectorizer(ngram_range=rango, min_df=1)
    X   = cv.fit_transform(corpus_demo)
    print(f"\n=== {nombre} ===")
    print(f"  n_features = {X.shape[1]}")
    print(f"  Vocabulario: {list(cv.get_feature_names_out())}")

print("\n→ Notar cómo 'primera guerra' y 'guerra mundial' son features en (1,2)")
print("→ 'primera guerra mundial' solo aparece como feature en (1,3)")



=== (1,1) — solo unigrams ===
  n_features = 13
  Vocabulario: ['el', 'es', 'excelente', 'fue', 'guerra', 'la', 'libro', 'mundial', 'primera', 'segunda', 'sobre', 'también', 'terrible']

=== (1,2) — uni + bigrams ===
  n_features = 28
  Vocabulario: ['el', 'el libro', 'es', 'es excelente', 'excelente', 'fue', 'fue terrible', 'guerra', 'guerra es', 'guerra mundial', 'la', 'la guerra', 'la primera', 'la segunda', 'libro', 'libro sobre', 'mundial', 'mundial fue', 'mundial también', 'primera', 'primera guerra', 'segunda', 'segunda guerra', 'sobre', 'sobre la', 'también', 'también fue', 'terrible']

=== (2,2) — solo bigrams ===
  n_features = 15
  Vocabulario: ['el libro', 'es excelente', 'fue terrible', 'guerra es', 'guerra mundial', 'la guerra', 'la primera', 'la segunda', 'libro sobre', 'mundial fue', 'mundial también', 'primera guerra', 'segunda guerra', 'sobre la', 'también fue']

=== (1,3) — uni+bi+trigrams ===
  n_features = 42
  Vocabulario: ['el', 'el libro', 'el libro sobre', 'es

---
# Parte 5 – TF-IDF

## El problema con Count Vectorizer

Si usamos solo frecuencias, palabras como "el", "de", "es" tienen conteos altísimos
en todos los documentos. No ayudan a distinguir géneros.

**TF-IDF** combina dos ideas:

- **TF** *(Term Frequency)*: qué tan frecuente es la palabra **en este documento**
- **IDF** *(Inverse Document Frequency)*: penaliza las palabras que aparecen
  en **todos** los documentos (poco útiles para distinguir)

Una palabra frecuente solo en algunos documentos → TF-IDF alto → útil para clasificar.  
Una palabra frecuente en todos los documentos → IDF bajo → peso bajo.

## Ejemplo

```
Corpus de 3 docs. La palabra "el" aparece en todos → IDF bajo → peso bajo.
La palabra "crónica" solo en 1 doc → IDF alto → peso alto.
```

## Parámetros importantes

| Parámetro | Qué hace |
|-----------|---------|
| `sublinear_tf=True` | Usa `log(1 + frecuencia)` en lugar de la frecuencia directa. Evita que documentos largos dominen por repetición. |
| `smooth_idf=True` | Evita errores matemáticos si un término aparece en todos los docs. Conviene dejarlo en `True`. |
| `use_idf=False` | Desactiva el IDF, quedando solo TF normalizado. |

> **Recomendación para empezar**: `TfidfVectorizer(sublinear_tf=True, min_df=2)` es
> un buen punto de partida para clasificación de texto.


In [ ]:
# ── Ejemplo interactivo: TF-IDF ──────────────────────────────────────────────
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd
import numpy as np

corpus_demo = [
    "el libro es bueno el libro es muy bueno",   # "libro" repetido
    "el libro tiene algunos errores graves",
    "el ensayo es brillante original y único",    # palabras únicas
]

tfidf = TfidfVectorizer()
X     = tfidf.fit_transform(corpus_demo)
feat  = tfidf.get_feature_names_out()

df_tfidf = pd.DataFrame(X.toarray().round(3),
                        columns=feat,
                        index=[f"d{i+1}" for i in range(len(corpus_demo))])
print("=== TF-IDF (smooth_idf=True, sublinear_tf=False) ===")
print(df_tfidf.to_string())

# Mostrar IDF de cada término
idf_series = pd.Series(tfidf.idf_, index=feat).sort_values()
print("\n=== IDF por término (menor = más común en el corpus) ===")
print(idf_series.round(3).to_string())
print("\n→ 'el' y 'es' tienen IDF bajo (aparecen en todos los docs)")
print("→ 'brillante', 'único', 'original' tienen IDF alto (aparecen en 1 solo doc)")

# Comparar sublinear_tf=False vs True
print("\n=== Efecto de sublinear_tf ===")
doc_largo  = ["guerra " * 50 + "economía inflación recesión"]
doc_corto  = ["guerra economía inflación recesión"]

for corpus_ex, label in [(doc_largo, "largo (guerra×50)"), (doc_corto, "corto (guerra×1)")]:
    for sublin, nombre in [(False, "sublinear=False"), (True, "sublinear=True")]:
        v = TfidfVectorizer(sublinear_tf=sublin)
        X_ex = v.fit_transform(corpus_ex)
        idx_guerra = list(v.get_feature_names_out()).index("guerra")
        print(f"  {label} | {nombre}: TF-IDF('guerra') = {X_ex[0, idx_guerra]:.4f}")


=== TF-IDF (smooth_idf=True, sublinear_tf=False) ===
    algunos  brillante  bueno     el  ensayo  errores     es  graves  libro    muy  original  tiene  único
d1    0.000      0.000  0.602  0.356   0.000    0.000  0.458   0.000  0.458  0.301     0.000  0.000  0.000
d2    0.451      0.000  0.000  0.266   0.000    0.451  0.000   0.451  0.343  0.000     0.000  0.451  0.000
d3    0.000      0.451  0.000  0.266   0.451    0.000  0.343   0.000  0.000  0.000     0.451  0.000  0.451

=== IDF por término (menor = más común en el corpus) ===
el           1.000
es           1.288
libro        1.288
brillante    1.693
bueno        1.693
ensayo       1.693
errores      1.693
algunos      1.693
graves       1.693
muy          1.693
original     1.693
tiene        1.693
único        1.693

→ 'el' y 'es' tienen IDF bajo (aparecen en todos los docs)
→ 'brillante', 'único', 'original' tienen IDF alto (aparecen en 1 solo doc)

=== Efecto de sublinear_tf ===
  largo (guerra×50) | sublinear=False: TF-IDF(

### Ejercicio 5 – TF-IDF: Palabras por Género

In [ ]:
def palabras_por_genero(df):
    """
    TODO:
    1. Vectorizar con TfidfVectorizer(sublinear_tf=True, max_features=5000, min_df=2)
    2. Calcular el TF-IDF promedio por género
    3. Graficar los 10 términos con mayor puntaje para cada género
    """
    pass

palabras_por_genero(df)


---
# Parte 6 – Hash Vectorizer

> Esta sección es **solo lectura**: entendé el concepto y ejecutá el ejemplo.

## ¿Cuándo Count y TF-IDF no alcanzan?

Ambos métodos necesitan construir un **vocabulario** antes de vectorizar.
Con millones de documentos o datos en tiempo real, esto puede ser un problema.

## La solución: el Hashing Trick

En vez de guardar `{"guerra": 42, "economía": 1337, ...}`, aplicamos una función
**hash** directamente a cada palabra para obtener su índice:

```python
indice = hash("guerra") % n_features   # siempre da el mismo número
```

No hay vocabulario que construir ni guardar.

```python
from sklearn.feature_extraction.text import HashingVectorizer

hv = HashingVectorizer(n_features=2**18, norm="l2")

# No necesita fit — transform directo
X = hv.transform(["el gato come pescado", "el perro come carne"])
```

## ¿Cuándo usarlo?

- Corpus muy grande que no cabe en memoria
- Datos en tiempo real (streaming)
- Cuando van a aparecer palabras nuevas en producción

## Parámetro clave: `n_features`

Define el tamaño fijo del vector. Con `2**18` ≈ 260 000 posiciones, las colisiones
(dos palabras distintas con el mismo índice) suelen ser despreciables.


In [ ]:
# ── Ejemplo interactivo: Hash Vectorizer ─────────────────────────────────────
from sklearn.feature_extraction.text import HashingVectorizer, CountVectorizer
import numpy as np

corpus_demo = [
    "el gato come pescado",
    "el perro come carne",
    "el gato y el perro juegan",
]

print("=== HashingVectorizer (n_features=2^5=32) ===")
hv = HashingVectorizer(n_features=2**5, norm="l2", alternate_sign=True)
X_hash = hv.transform(corpus_demo)   # ¡No requiere fit!
print(f"Shape:     {X_hash.shape}")
print(f"NNZ total: {X_hash.nnz}")
print(f"NNZ/doc:   {X_hash.nnz / len(corpus_demo):.1f}")

print("\n→ No hay vocabulario almacenado (get_feature_names_out no existe)")
print(f"→ El mismo vectorizador sirve para cualquier texto nuevo sin reentrenar")

# Comparar dimensionalidad con CountVectorizer
cv = CountVectorizer()
X_cv = cv.fit_transform(corpus_demo)
print(f"\nCountVectorizer:  shape = {X_cv.shape}  (vocab = {len(cv.vocabulary_)} términos)")
print(f"HashVectorizer:   shape = {X_hash.shape}  (n_features fijo = 32)")

# Demostrar que no necesita fit: texto completamente nuevo
nuevo_texto = ["zorzal cantó entre los ceibos del litoral"]
X_nuevo = hv.transform(nuevo_texto)  # funciona sin reentrenar
print(f"\nTexto nuevo sin reentrenar: shape = {X_nuevo.shape}")

# Mostrar el problema de alternate_sign con MultinomialNB
print("\n=== alternate_sign y compatibilidad con Naive Bayes ===")
hv_pos = HashingVectorizer(n_features=2**10, norm=None, alternate_sign=False)
hv_alt = HashingVectorizer(n_features=2**10, norm=None, alternate_sign=True)
X_pos  = hv_pos.transform(corpus_demo)
X_alt  = hv_alt.transform(corpus_demo)
print(f"alternate_sign=False → valores negativos: {(X_pos.data < 0).sum()} (cero → OK para MultinomialNB)")
print(f"alternate_sign=True  → valores negativos: {(X_alt.data < 0).sum()} (algunos negativos → NO usar con MultinomialNB)")


=== HashingVectorizer (n_features=2^5=32) ===
Shape:     (3, 32)
NNZ total: 12
NNZ/doc:   4.0

→ No hay vocabulario almacenado (get_feature_names_out no existe)
→ El mismo vectorizador sirve para cualquier texto nuevo sin reentrenar

CountVectorizer:  shape = (3, 7)  (vocab = 7 términos)
HashVectorizer:   shape = (3, 32)  (n_features fijo = 32)

Texto nuevo sin reentrenar: shape = (1, 32)

=== alternate_sign y compatibilidad con Naive Bayes ===
alternate_sign=False → valores negativos: 0 (cero → OK para MultinomialNB)
alternate_sign=True  → valores negativos: 11 (algunos negativos → NO usar con MultinomialNB)


---
# Parte 7 – Comparación de Todos los Métodos

Hasta ahora aprendimos cada vectorizador por separado. Ahora los comparamos
con el mismo clasificador y la misma evaluación para ver cuál funciona mejor.

## Representaciones a comparar

| ID | Método | Configuración |
|----|--------|--------------|
| A | One-Hot | `binary=True, max_features=5000` |
| B | Count | `max_features=5000, min_df=2` |
| C | TF-IDF base | `max_features=5000, min_df=2` |
| D | TF-IDF sublinear | `sublinear_tf=True, max_features=5000` |
| E | TF-IDF bigramas | `ngram_range=(1,2), sublinear_tf=True` |

Clasificador: **Logistic Regression** con 5-fold cross-validation.


### Ejercicio 6 – Comparar Vectorizadores

In [ ]:
def comparar_vectorizadores(df):
    """
    TODO:
    1. Crear los 5 vectorizadores:
       - One-Hot  (CountVectorizer binary=True, max_features=5000)
       - Count    (CountVectorizer max_features=5000, min_df=2)
       - TF-IDF base    (max_features=5000, min_df=2)
       - TF-IDF sublin  (sublinear_tf=True, max_features=5000, min_df=2)
       - TF-IDF bigrams (ngram_range=(1,2), sublinear_tf=True, max_features=8000)
    2. Evaluar cada uno con LogisticRegression (accuracy, cv=5)
    3. Mostrar tabla y gráfico de barras ordenado de mejor a peor
    """
    pass

comparar_vectorizadores(df)


### Ejercicio 7 – Evaluar el Mejor Modelo

In [ ]:
def evaluar_mejor_modelo(df):
    """
    TODO:
    1. Usar TF-IDF bigramas + LogisticRegression, split 80/20 (stratify=y)
    2. Mostrar la matriz de confusión normalizada
    3. Mostrar el classification report
    """
    pass

evaluar_mejor_modelo(df)


---
# Parte 8 – Visualización con PCA

Después del benchmark, es útil ver visualmente si los géneros se separan en el
espacio de representación. Usamos **TruncatedSVD** para reducir miles de dimensiones
a solo 2, y hacemos un scatter plot.

```python
from sklearn.decomposition import TruncatedSVD

svd  = TruncatedSVD(n_components=2, random_state=42)
X_2d = svd.fit_transform(X_sparse)   # pasa de (n_docs, n_features) a (n_docs, 2)
```

Si los puntos de cada género forman grupos separados, la representación es más útil.

> Los primeros 2 componentes suelen capturar solo el 10–15% de la varianza total.
> Que los puntos se superpongan no significa que el método sea malo — la separación
> puede existir en otras dimensiones que no podemos ver en 2D.


### Ejercicio 8 – Visualización 2D con PCA

In [ ]:
def visualizar_en_2d(df):
    """
    TODO:
    1. Vectorizar con TF-IDF base y TF-IDF bigramas
    2. Reducir a 2 dimensiones con TruncatedSVD(n_components=2)
    3. Hacer un scatter plot coloreado por género para cada representación
    """
    pass

visualizar_en_2d(df)
